<a href="https://colab.research.google.com/github/fajriantomanungki/lomba_kti/blob/main/Data/gabunagData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

In [12]:
BASE_DIR = Path("/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/output/pakai")

print("BASE_DIR:", BASE_DIR)
print("Folder ada?", BASE_DIR.exists())

print("\nIsi folder pakai:")
for f in BASE_DIR.glob("*"):
    print("-", f.name)

BASE_DIR: /content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/output/pakai
Folder ada? True

Isi folder pakai:
- ntl_sulampua_2025_final.csv
- ntl_sulampua_2025_final.geojson
- spbu_density_sulampua.csv
- spbu_density_sulampua_calculated.csv
- spbu_coverage_luas_kabkot_sulampua.csv
- spbu_coverage_luas_kabkot_sulampua.geojson
- spbu_coverage_sulampua_calculated.csv
- road_density_sulampua.csv
- road_density_sulampua.geojson
- road_connectivity_index_sulampua.csv
- aeds_road_density_connectivity_sulampua.csv
- aeds_road_density_connectivity_sulampua.geojson


In [13]:
files = {
    "spbu_density": "spbu_density_sulampua_calculated.csv",
    "road_connectivity": "aeds_road_density_connectivity_sulampua.csv",
    "ntl": "ntl_sulampua_2025_final.csv",
    "connectivity_index": "road_connectivity_index_sulampua.csv",
    "road_density": "road_density_sulampua.csv",
    "spbu_coverage": "spbu_coverage_sulampua_calculated.csv"
}

dfs = {}

for key, filename in files.items():
    path = BASE_DIR / filename

    if not path.exists():
        print(f"❌ File tidak ditemukan: {filename}")
        continue

    df = pd.read_csv(path)
    dfs[key] = df
    print(f"✅ {key}: {df.shape}")
    print(df.columns.tolist())
    print()

✅ spbu_density: (133, 5)
['NAME_1', 'NAME_2', 'jumlah_spbu', 'luas_km2', 'spbu_density']

✅ road_connectivity: (134, 30)
['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'provinsi', 'NL_NAME_1', 'kabupaten_kota', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'luas_km2', 'road_length_km', 'road_density_km_per_km2', 'road_density_km_per_100km2', 'nodes', 'edges', 'components', 'largest_component_edges', 'largest_component_ratio', 'beta_index', 'gamma_index', 'alpha_index', 'beta_norm', 'gamma_norm', 'alpha_norm', 'lcr_norm', 'connectivity_index']

✅ ntl: (134, 21)
['ntl_mean', 'ntl_sum', 'ntl_max', 'ntl_median', 'system:index', 'ENGTYPE_2', 'NL_NAME_2', 'HASC_2', 'NL_NAME_1', 'CC_2', 'NAME_2', 'NAME_1', 'TYPE_2', 'COUNTRY', 'GID_0', 'GID_1', 'GID_2', 'VARNAME_2', 'ntl_index_0_100', 'ntl_sum_index_0_100', 'ntl_dimension_index']

✅ connectivity_index: (134, 15)
['nodes', 'edges', 'components', 'largest_component_edges', 'largest_component_ratio', 'beta_index', 'gamma_index', 'al

In [14]:
def clean_text(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("Kabupaten ", "")
    x = x.replace("Kota ", "")
    x = x.replace("Kab. ", "")
    x = x.replace("KABUPATEN ", "")
    x = x.replace("KOTA ", "")
    x = x.title()

    # koreksi umum
    replacements = {
        "Adm. Kep. Seribu": "Kepulauan Seribu",
        "Kep.": "Kepulauan",
        "Kep ": "Kepulauan ",
    }

    for old, new in replacements.items():
        x = x.replace(old, new)

    return x.strip()


def standardize_columns(df):
    df = df.copy()

    # ubah nama kolom ke lowercase agar mudah
    df.columns = [c.strip().lower() for c in df.columns]

    # mapping nama kolom provinsi
    prov_candidates = ["provinsi", "province", "name_1", "nama_provinsi"]
    kab_candidates = ["kabupaten_kota", "kab_kota", "kabupaten", "kota", "name_2", "nama_kabupaten", "nama_kabupaten_kota"]

    prov_col = None
    kab_col = None

    for c in prov_candidates:
        if c in df.columns:
            prov_col = c
            break

    for c in kab_candidates:
        if c in df.columns:
            kab_col = c
            break

    if prov_col is None:
        raise ValueError(f"Kolom provinsi tidak ditemukan. Kolom tersedia: {df.columns.tolist()}")

    if kab_col is None:
        raise ValueError(f"Kolom kabupaten/kota tidak ditemukan. Kolom tersedia: {df.columns.tolist()}")

    df = df.rename(columns={
        prov_col: "provinsi",
        kab_col: "kabupaten_kota"
    })

    df["provinsi"] = df["provinsi"].apply(clean_text)
    df["kabupaten_kota"] = df["kabupaten_kota"].apply(clean_text)

    return df

In [15]:
for key in list(dfs.keys()):
    dfs[key] = standardize_columns(dfs[key])
    print(f"✅ {key}")
    print(dfs[key][["provinsi", "kabupaten_kota"]].head())
    print()

✅ spbu_density
    provinsi  kabupaten_kota
0  Gorontalo         Boalemo
1  Gorontalo     Bonebolango
2  Gorontalo       Gorontalo
3  Gorontalo  Gorontaloutara
4  Gorontalo   Kotagorontalo

✅ road_connectivity
    provinsi  kabupaten_kota
0  Gorontalo         Boalemo
1  Gorontalo     Bonebolango
2  Gorontalo    Danaulimboto
3  Gorontalo       Gorontalo
4  Gorontalo  Gorontaloutara

✅ ntl
    provinsi  kabupaten_kota
0  Gorontalo         Boalemo
1  Gorontalo     Bonebolango
2  Gorontalo    Danaulimboto
3  Gorontalo       Gorontalo
4  Gorontalo  Gorontaloutara

✅ connectivity_index
    provinsi  kabupaten_kota
0  Gorontalo         Boalemo
1  Gorontalo     Bonebolango
2  Gorontalo    Danaulimboto
3  Gorontalo       Gorontalo
4  Gorontalo  Gorontaloutara

✅ road_density
    provinsi  kabupaten_kota
0  Gorontalo         Boalemo
1  Gorontalo     Bonebolango
2  Gorontalo    Danaulimboto
3  Gorontalo       Gorontalo
4  Gorontalo  Gorontaloutara

✅ spbu_coverage
    provinsi  kabupaten_kota
0  

In [16]:
for key, df in dfs.items():
    dup = df[df.duplicated(subset=["provinsi", "kabupaten_kota"], keep=False)]

    print(f"{key}:")
    print("Jumlah duplikat:", len(dup))

    if len(dup) > 0:
        display(dup[["provinsi", "kabupaten_kota"]].sort_values(["provinsi", "kabupaten_kota"]))

    print("-" * 50)

spbu_density:
Jumlah duplikat: 0
--------------------------------------------------
road_connectivity:
Jumlah duplikat: 0
--------------------------------------------------
ntl:
Jumlah duplikat: 0
--------------------------------------------------
connectivity_index:
Jumlah duplikat: 0
--------------------------------------------------
road_density:
Jumlah duplikat: 0
--------------------------------------------------
spbu_coverage:
Jumlah duplikat: 0
--------------------------------------------------


In [17]:
for key in list(dfs.keys()):
    dfs[key] = dfs[key].drop_duplicates(subset=["provinsi", "kabupaten_kota"], keep="first")

In [18]:
master_key = "road_connectivity"

master = dfs[master_key][["provinsi", "kabupaten_kota"]].copy()

print("Jumlah wilayah master:", len(master))
display(master.head())

Jumlah wilayah master: 134


,provinsi,kabupaten_kota
0,Gorontalo,Boalemo
1,Gorontalo,Bonebolango
2,Gorontalo,Danaulimboto
3,Gorontalo,Gorontalo
4,Gorontalo,Gorontaloutara


In [19]:
def safe_merge(master, df, name):
    df = df.copy()

    # buang kolom duplikat selain key jika ada suffix lama
    df = df.loc[:, ~df.columns.duplicated()]

    before = len(master)

    merged = master.merge(
        df,
        on=["provinsi", "kabupaten_kota"],
        how="left",
        suffixes=("", f"_{name}")
    )

    after = len(merged)

    if before != after:
        print(f"⚠️ Jumlah baris berubah saat merge {name}: {before} -> {after}")
    else:
        print(f"✅ Merge {name} aman: {after} baris")

    # cek wilayah yang tidak match
    indicator_cols = [c for c in df.columns if c not in ["provinsi", "kabupaten_kota"]]

    if indicator_cols:
        first_col = indicator_cols[0]
        missing = merged[merged[first_col].isna()][["provinsi", "kabupaten_kota"]]
        print(f"Wilayah tidak match dari {name}: {len(missing)}")

        if len(missing) > 0:
            display(missing)

    print("-" * 60)

    return merged

In [20]:
final = master.copy()

for key, df in dfs.items():
    if key == master_key:
        # merge isi master juga, bukan cuma key
        final = safe_merge(final, df, key)
    else:
        final = safe_merge(final, df, key)

print("Ukuran final:", final.shape)
display(final.head())

✅ Merge spbu_density aman: 134 baris
Wilayah tidak match dari spbu_density: 1


,provinsi,kabupaten_kota
2,Gorontalo,Danaulimboto


------------------------------------------------------------
✅ Merge road_connectivity aman: 134 baris
Wilayah tidak match dari road_connectivity: 0
------------------------------------------------------------
✅ Merge ntl aman: 134 baris
Wilayah tidak match dari ntl: 0
------------------------------------------------------------
✅ Merge connectivity_index aman: 134 baris
Wilayah tidak match dari connectivity_index: 0
------------------------------------------------------------
✅ Merge road_density aman: 134 baris
Wilayah tidak match dari road_density: 0
------------------------------------------------------------
✅ Merge spbu_coverage aman: 134 baris
Wilayah tidak match dari spbu_coverage: 1


,provinsi,kabupaten_kota
2,Gorontalo,Danaulimboto


------------------------------------------------------------
Ukuran final: (134, 87)


,provinsi,kabupaten_kota,jumlah_spbu,luas_km2,spbu_density,gid_2,gid_0,country,gid_1,nl_name_1,...,road_length_km_road_density,road_density_km_per_km2_road_density,road_density_km_per_100km2_road_density,jumlah_spbu_spbu_coverage,luas_km2_spbu_coverage,spbu_density_spbu_coverage,coverage_spbu_per_km2,coverage_spbu_per_1000km2,kelas_coverage_spbu,kelas_coverage_spbu_absolut
0,Gorontalo,Boalemo,4.0,1856.92,0.002154,IDN.6.1_1,IDN,Indonesia,IDN.6_1,NaN,...,1022.725057,0.554549,55.454911,4.0,1856.92,0.002154,0.002154,2.154105,Tinggi,Rendah
1,Gorontalo,Bonebolango,1.0,1917.76,0.000521,IDN.6.2_1,IDN,Indonesia,IDN.6_1,NaN,...,827.493944,0.434437,43.443750,1.0,1917.76,0.000521,0.000521,0.521442,Rendah,Sangat Rendah
2,Gorontalo,Danaulimboto,NaN,NaN,NaN,IDN.6.3_1,IDN,Indonesia,IDN.6_1,NaN,...,9.965330,0.368264,36.826407,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gorontalo,Gorontalo,12.0,2147.57,0.005588,IDN.6.5_1,IDN,Indonesia,IDN.6_1,NaN,...,2379.452008,1.115605,111.560545,12.0,2147.57,0.005588,0.005588,5.587711,Tinggi,Tinggi
4,Gorontalo,Gorontaloutara,1.0,1717.78,0.000582,IDN.6.4_1,IDN,Indonesia,IDN.6_1,NaN,...,735.817303,0.431344,43.134414,1.0,1717.78,0.000582,0.000582,0.582147,Rendah,Sangat Rendah


In [21]:
final = final.dropna(axis=1, how="all")

# hapus kolom duplikat nama
final = final.loc[:, ~final.columns.duplicated()]

print("Ukuran final setelah dirapikan:", final.shape)
print(final.columns.tolist())

Ukuran final setelah dirapikan: (134, 78)
['provinsi', 'kabupaten_kota', 'jumlah_spbu', 'luas_km2', 'spbu_density', 'gid_2', 'gid_0', 'country', 'gid_1', 'type_2', 'engtype_2', 'cc_2', 'hasc_2', 'luas_km2_road_connectivity', 'road_length_km', 'road_density_km_per_km2', 'road_density_km_per_100km2', 'nodes', 'edges', 'components', 'largest_component_edges', 'largest_component_ratio', 'beta_index', 'gamma_index', 'alpha_index', 'beta_norm', 'gamma_norm', 'alpha_norm', 'lcr_norm', 'connectivity_index', 'ntl_mean', 'ntl_sum', 'ntl_max', 'ntl_median', 'system:index', 'engtype_2_ntl', 'hasc_2_ntl', 'cc_2_ntl', 'type_2_ntl', 'country_ntl', 'gid_0_ntl', 'gid_1_ntl', 'gid_2_ntl', 'ntl_index_0_100', 'ntl_sum_index_0_100', 'ntl_dimension_index', 'nodes_connectivity_index', 'edges_connectivity_index', 'components_connectivity_index', 'largest_component_edges_connectivity_index', 'largest_component_ratio_connectivity_index', 'beta_index_connectivity_index', 'gamma_index_connectivity_index', 'alpha_

In [22]:
missing_summary = pd.DataFrame({
    "kolom": final.columns,
    "jumlah_missing": final.isna().sum().values,
    "persen_missing": (final.isna().sum().values / len(final) * 100).round(2)
}).sort_values("jumlah_missing", ascending=False)

display(missing_summary)

,kolom,jumlah_missing,persen_missing
3,luas_km2,1,0.75
2,jumlah_spbu,1,0.75
4,spbu_density,1,0.75
75,coverage_spbu_per_1000km2,1,0.75
74,coverage_spbu_per_km2,1,0.75
...,...,...,...
66,hasc_2_road_density,0,0.00
65,cc_2_road_density,0,0.00
64,engtype_2_road_density,0,0.00
63,type_2_road_density,0,0.00


In [23]:
rows_with_missing = final[final.isna().any(axis=1)].copy()

print("Jumlah wilayah dengan minimal satu missing value:", len(rows_with_missing))

display(
    rows_with_missing[["provinsi", "kabupaten_kota"]]
    .sort_values(["provinsi", "kabupaten_kota"])
)

Jumlah wilayah dengan minimal satu missing value: 1


,provinsi,kabupaten_kota
2,Gorontalo,Danaulimboto


In [24]:
output_csv = BASE_DIR / "aeds_sulampua_variabel_gabungan_kabkota.csv"
output_excel = BASE_DIR / "aeds_sulampua_variabel_gabungan_kabkota.xlsx"
validation_csv = BASE_DIR / "aeds_merge_validation_report.csv"

final.to_csv(output_csv, index=False)
final.to_excel(output_excel, index=False)
missing_summary.to_csv(validation_csv, index=False)

print("✅ File berhasil disimpan:")
print(output_csv)
print(output_excel)
print(validation_csv)

✅ File berhasil disimpan:
/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/output/pakai/aeds_sulampua_variabel_gabungan_kabkota.csv
/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/output/pakai/aeds_sulampua_variabel_gabungan_kabkota.xlsx
/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/output/pakai/aeds_merge_validation_report.csv


In [25]:
display(final.head(20))
print(final.shape)

,provinsi,kabupaten_kota,jumlah_spbu,luas_km2,spbu_density,gid_2,gid_0,country,gid_1,type_2,...,road_length_km_road_density,road_density_km_per_km2_road_density,road_density_km_per_100km2_road_density,jumlah_spbu_spbu_coverage,luas_km2_spbu_coverage,spbu_density_spbu_coverage,coverage_spbu_per_km2,coverage_spbu_per_1000km2,kelas_coverage_spbu,kelas_coverage_spbu_absolut
0,Gorontalo,Boalemo,4.0,1856.92,0.002154,IDN.6.1_1,IDN,Indonesia,IDN.6_1,Kabupaten,...,1022.725057,0.554549,55.454911,4.0,1856.92,0.002154,0.002154,2.154105,Tinggi,Rendah
1,Gorontalo,Bonebolango,1.0,1917.76,0.000521,IDN.6.2_1,IDN,Indonesia,IDN.6_1,Kabupaten,...,827.493944,0.434437,43.443750,1.0,1917.76,0.000521,0.000521,0.521442,Rendah,Sangat Rendah
2,Gorontalo,Danaulimboto,NaN,NaN,NaN,IDN.6.3_1,IDN,Indonesia,IDN.6_1,Kabupaten,...,9.965330,0.368264,36.826407,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gorontalo,Gorontalo,12.0,2147.57,0.005588,IDN.6.5_1,IDN,Indonesia,IDN.6_1,Kabupaten,...,2379.452008,1.115605,111.560545,12.0,2147.57,0.005588,0.005588,5.587711,Tinggi,Tinggi
4,Gorontalo,Gorontaloutara,1.0,1717.78,0.000582,IDN.6.4_1,IDN,Indonesia,IDN.6_1,Kabupaten,...,735.817303,0.431344,43.134414,1.0,1717.78,0.000582,0.000582,0.582147,Rendah,Sangat Rendah
5,Gorontalo,Kotagorontalo,17.0,65.58,0.259225,IDN.6.6_1,IDN,Indonesia,IDN.6_1,Kota,...,563.716787,8.653895,865.389526,17.0,65.58,0.259225,0.259225,259.225374,Sangat Tinggi,Sangat Tinggi
6,Gorontalo,Pohuwato,6.0,4391.01,0.001366,IDN.6.7_1,IDN,Indonesia,IDN.6_1,Kabupaten,...,956.461681,0.219323,21.932270,6.0,4391.01,0.001366,0.001366,1.366428,Sedang,Rendah
7,Maluku,Ambon,17.0,307.44,0.055295,IDN.19.1_1,IDN,Indonesia,IDN.19_1,Kota,...,756.028254,2.485782,248.578220,17.0,307.44,0.055295,0.055295,55.295342,Sangat Tinggi,Sangat Tinggi
8,Maluku,Buru,4.0,4504.85,0.000888,IDN.19.3_1,IDN,Indonesia,IDN.19_1,Kabupaten,...,1210.933732,0.271509,27.150936,4.0,4504.85,0.000888,0.000888,0.887932,Sedang,Sangat Rendah
9,Maluku,Buruselatan,3.0,4317.28,0.000695,IDN.19.2_1,IDN,Indonesia,IDN.19_1,Kabupaten,...,725.715653,0.169884,16.988394,3.0,4317.28,0.000695,0.000695,0.694882,Rendah,Sangat Rendah


(134, 78)
